# Human Detection with Activity Analysis

Detects people in an image and classifies each person's activity from YOLOv8 pose keypoints. Built for natural-disaster / search-and-rescue scenes. Run top-to-bottom in Google Colab; it will prompt you to upload an image and then download the annotated result + activity report.

**Models:** `yolov8n.pt` (person detection) + `yolov8n-pose.pt` (pose).

In [ ]:
# ========================================
# HUMAN DETECTION WITH ACTIVITY ANALYSIS
# Fixed Version - Detects Humans + HEIC Support
# ========================================

# Install dependencies
print("📦 Installing dependencies...")
!pip install -q pillow opencv-python-headless ultralytics pillow-heif pillow-avif-plugin

print("✅ Dependencies installed!\n")

# Import libraries
print("📚 Importing libraries...")
from PIL import Image
import os
import cv2
import numpy as np
from google.colab import files
from IPython.display import display
from ultralytics import YOLO
import pillow_heif

# Register HEIF opener
pillow_heif.register_heif_opener()

# Try to register AVIF support
try:
    import pillow_avif
except:
    pass

print("✅ Libraries imported!\n")

# Load YOLO model for person detection
print("🤖 Loading YOLO models...")
# Use YOLOv8 for person detection
detection_model = YOLO('yolov8n.pt')
# Use YOLOv8-pose for activity analysis
pose_model = YOLO('yolov8n-pose.pt')
print("✅ Models loaded!\n")

def convert_heic_to_jpg(heic_path):
    """Convert HEIC to JPG"""
    try:
        img = Image.open(heic_path)
        jpg_path = heic_path.rsplit('.', 1)[0] + '_converted.jpg'
        rgb_img = img.convert('RGB')
        rgb_img.save(jpg_path, 'JPEG', quality=95)
        return jpg_path
    except Exception as e:
        print(f"❌ HEIC conversion error: {e}")
        return None

def convert_to_supported_format(image_path):
    """Convert any image format to JPG for YOLO compatibility"""
    try:
        # Check if file exists
        if not os.path.exists(image_path):
            print(f"❌ File not found: {image_path}")
            return None

        # Get file extension
        ext = image_path.lower().split('.')[-1]

        # If already a supported format, return as-is
        if ext in ['jpg', 'jpeg', 'png']:
            return image_path

        # Otherwise convert to JPG
        print(f"🔄 Converting {ext.upper()} to JPG...")
        img = Image.open(image_path)
        jpg_path = image_path.rsplit('.', 1)[0] + '_converted.jpg'

        # Convert to RGB (removes alpha channel if present)
        if img.mode in ('RGBA', 'LA', 'P'):
            rgb_img = img.convert('RGB')
        else:
            rgb_img = img

        rgb_img.save(jpg_path, 'JPEG', quality=95)
        print(f"✅ Converted to: {jpg_path}")
        return jpg_path
    except Exception as e:
        print(f"❌ Conversion error: {e}")
        return None

def analyze_activity_from_keypoints(keypoints):
    """
    Analyze activity based on YOLO pose keypoints
    Keypoints: nose, eyes, ears, shoulders, elbows, wrists, hips, knees, ankles
    """
    try:
        if keypoints is None or len(keypoints) < 17:
            return "Person detected"

        # Extract key body parts (x, y, confidence)
        nose = keypoints[0]
        l_shoulder = keypoints[5]
        r_shoulder = keypoints[6]
        l_wrist = keypoints[9]
        r_wrist = keypoints[10]
        l_hip = keypoints[11]
        r_hip = keypoints[12]
        l_knee = keypoints[13]
        r_knee = keypoints[14]

        # Check if we have enough visible keypoints
        visible_keypoints = sum([1 for kp in keypoints if kp[2] > 0.3])
        if visible_keypoints < 5:
            return "Person detected"

        # Calculate body orientation
        shoulder_center_y = (l_shoulder[1] + r_shoulder[1]) / 2
        hip_center_y = (l_hip[1] + r_hip[1]) / 2
        body_height = abs(shoulder_center_y - hip_center_y)

        # Lying down: shoulders and hips at similar height
        if body_height < 30:
            return "Lying down"

        # Sitting: knees visible and higher than hips
        if l_knee[2] > 0.3 and r_knee[2] > 0.3:
            knee_avg_y = (l_knee[1] + r_knee[1]) / 2
            if knee_avg_y < hip_center_y + 20:
                return "Sitting"

        # Arms raised: wrists higher than shoulders
        if l_wrist[2] > 0.3 and l_wrist[1] < l_shoulder[1] - 30:
            return "Arms raised"
        if r_wrist[2] > 0.3 and r_wrist[1] < r_shoulder[1] - 30:
            return "Arms raised"

        # Using phone/device: hands near face
        if l_wrist[2] > 0.3 and abs(l_wrist[1] - nose[1]) < 80:
            return "Using device"
        if r_wrist[2] > 0.3 and abs(r_wrist[1] - nose[1]) < 80:
            return "Using device"

        # Walking: significant leg separation
        if l_knee[2] > 0.3 and r_knee[2] > 0.3:
            leg_separation = abs(l_knee[1] - r_knee[1])
            if leg_separation > 60:
                return "Walking"

        return "Standing"

    except Exception as e:
        print(f"⚠️ Activity analysis error: {e}")
        return "Person detected"

def draw_results(image, bbox, activity, person_num, keypoints=None):
    """Draw bounding box, activity label, and pose skeleton"""
    x1, y1, x2, y2 = map(int, bbox[:4])

    # Draw bounding box with thick green line
    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 4)

    # Draw pose skeleton if available
    if keypoints is not None and len(keypoints) >= 17:
        # Define skeleton connections
        skeleton = [
            (5, 7), (7, 9),   # Left arm
            (6, 8), (8, 10),  # Right arm
            (5, 6),           # Shoulders
            (5, 11), (6, 12), # Torso
            (11, 12),         # Hips
            (11, 13), (13, 15), # Left leg
            (12, 14), (14, 16)  # Right leg
        ]

        # Draw skeleton lines
        for connection in skeleton:
            pt1_idx, pt2_idx = connection
            pt1 = keypoints[pt1_idx]
            pt2 = keypoints[pt2_idx]

            if pt1[2] > 0.3 and pt2[2] > 0.3:  # Check confidence
                cv2.line(image,
                        (int(pt1[0]), int(pt1[1])),
                        (int(pt2[0]), int(pt2[1])),
                        (255, 0, 0), 3)

        # Draw keypoints
        for point in keypoints:
            if point[2] > 0.3:  # Check confidence
                cv2.circle(image, (int(point[0]), int(point[1])), 5, (0, 0, 255), -1)

    # Prepare label
    label = f"Person {person_num}: {activity}"
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1.0
    thickness = 2

    # Get text size
    (text_w, text_h), baseline = cv2.getTextSize(label, font, font_scale, thickness)

    # Position above bbox
    text_x = x1
    text_y = max(text_h + 20, y1 - 15)

    # Draw background rectangle
    cv2.rectangle(image,
                  (text_x - 10, text_y - text_h - 10),
                  (text_x + text_w + 10, text_y + baseline + 10),
                  (0, 0, 0), -1)

    # Draw text in bright green
    cv2.putText(image, label, (text_x, text_y), font, font_scale, (0, 255, 0), thickness)

    return image

# Main processing
print("="*60)
print("📤 UPLOAD YOUR IMAGE")
print("   Supported: JPG, PNG, HEIC, AVIF, WEBP, BMP, TIFF")
print("="*60)
uploaded = files.upload()

if not uploaded:
    print("❌ No file uploaded!")
else:
    original_file_path = list(uploaded.keys())[0]
    print(f"✅ Uploaded: {original_file_path}\n")

    # Convert to supported format if needed
    print("🔄 Checking image format...")
    file_path = convert_to_supported_format(original_file_path)

    if file_path is None:
        print("❌ Could not process the image file.")
    else:
        print(f"✅ Using: {file_path}\n")

    if file_path is None:
        print("❌ Could not process the image file.")
    else:
        print(f"✅ Using: {file_path}\n")

        # Load image
        print("🖼️ Loading image...")
        img = cv2.imread(file_path)

        if img is None:
            print("❌ Could not load image with OpenCV!")
            print("Trying alternative loading method...")
            try:
                pil_img = Image.open(file_path)
                img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
                print("✅ Image loaded successfully!\n")
            except Exception as e:
                print(f"❌ Failed to load image: {e}")
                img = None
        else:
            print("✅ Image loaded!\n")

        if img is not None:
            img_height, img_width = img.shape[:2]
            print(f"📐 Image size: {img_width}x{img_height}")

            # Run person detection first
            print("\n🔍 Step 1: Detecting persons...")
            detection_results = detection_model(file_path, conf=0.25, verbose=False)

        # Filter only person detections (class 0)
        result = detection_results[0]
        boxes = result.boxes

        # Get only person class (class 0)
        person_indices = [i for i, cls in enumerate(boxes.cls) if int(cls) == 0]
        person_count = len(person_indices)

        print(f"✅ Found {person_count} person(s)!\n")

        if person_count == 0:
            print("⚠️ No persons detected in the image.")
            print("💡 Tips:")
            print("  - Make sure there are clearly visible people in the image")
            print("  - Try an image with better lighting")
            print("  - Ensure people are not too small in the frame")
            output_path = "result_no_persons.jpg"
            cv2.imwrite(output_path, img)
            activity_results = []
        else:
            print("🎯 Step 2: Analyzing poses and activities...\n")

            # Now run pose detection
            pose_results = pose_model(file_path, conf=0.25, verbose=False)
            pose_result = pose_results[0]

            annotated_img = img.copy()
            activity_results = []

            # Process each detected person
            for idx, i in enumerate(person_indices):
                bbox = boxes.xyxy[i].cpu().numpy()
                confidence = float(boxes.conf[i].cpu().numpy())

                # Get keypoints for this person if available
                keypoints = None
                if hasattr(pose_result, 'keypoints') and len(pose_result.keypoints) > idx:
                    try:
                        kpts_xy = pose_result.keypoints.xy[idx].cpu().numpy()
                        kpts_conf = pose_result.keypoints.conf[idx].cpu().numpy()
                        keypoints = np.column_stack([kpts_xy, kpts_conf.reshape(-1, 1)])
                    except:
                        keypoints = None

                # Analyze activity
                activity = analyze_activity_from_keypoints(keypoints)

                # Store results
                activity_results.append({
                    'person_id': idx + 1,
                    'activity': activity,
                    'confidence': confidence
                })

                # Draw on image
                annotated_img = draw_results(annotated_img, bbox, activity, idx + 1, keypoints)

                print(f"👤 Person {idx+1}: {activity} (Confidence: {confidence:.2%})")

            output_path = "result_with_activities.jpg"
            cv2.imwrite(output_path, annotated_img)
            print(f"\n✅ Analysis complete!\n")

        # Display
        print("="*60)
        print("🎨 RESULT IMAGE")
        print("="*60)
        display(Image.open(output_path))

        # Save report
        summary_path = "activity_report.txt"
        with open(summary_path, 'w') as f:
            f.write("="*60 + "\n")
            f.write("HUMAN ACTIVITY DETECTION REPORT\n")
            f.write("="*60 + "\n\n")
            f.write(f"Original file: {original_file_path}\n")
            f.write(f"Processed file: {file_path}\n")
            f.write(f"Size: {img_width}x{img_height}\n")
            f.write(f"Total persons detected: {person_count}\n\n")

            if activity_results:
                f.write("INDIVIDUAL ANALYSIS:\n")
                f.write("-"*40 + "\n")
                for result in activity_results:
                    f.write(f"\nPerson {result['person_id']}:\n")
                    f.write(f"  Activity: {result['activity']}\n")
                    f.write(f"  Detection Confidence: {result['confidence']:.2%}\n")
            else:
                f.write("No persons detected.\n")
                f.write("\nPossible reasons:\n")
                f.write("- People are too small or far away\n")
                f.write("- Poor lighting conditions\n")
                f.write("- People are partially occluded\n")

        # Download
        print("\n📥 DOWNLOADING FILES...")
        files.download(output_path)
        files.download(summary_path)

        print("\n✅ ALL DONE! 🎉")
        if person_count > 0:
            print("🎁 Check the image for pose skeleton visualization!")